In [70]:
import os
from pathlib import Path

import pandas as pd
import geopandas as gpd

In [15]:
data_path = Path(os.environ["DATA_PATH"])
generated_path = data_path / "generated"
ghsl_path = Path(os.environ["GHSL_PATH"])

In [ ]:
df_polygons = gpd.read_file(
    generated_path / "polygons" / "renamed_with_nearby" / "200_300" / "2020.gpkg",
)
df_features = gpd.read_file(generated_path / "features" / "filtered_places.gpkg")

In [83]:
name_count = df_polygons["name"].apply(
    lambda x: len(x.split("+")) if not pd.isna(x) else 0,
)
single_names = df_polygons.loc[name_count <= 1, "name"]
multiple_names = df_polygons.loc[name_count > 1, ["polygon_id", "name"]]

In [89]:
(
    multiple_names
    .assign(
        feature_id=lambda df: df["name"].str.split("+").apply(lambda name_list: [name.split(" ")[-1].strip("[]") for name in name_list])
    )
    .drop(columns=["name"])
    .explode("feature_id")
    .merge(df_features[["feature_id", "feature_pop", "name"]], on="feature_id", how="left")
    .sort_values(["polygon_id", "feature_pop"], ascending=[True, False])
    .drop_duplicates(subset=["polygon_id"], keep="first")
)

,polygon_id,feature_id,feature_pop,name
0,p001165,f0066753,2.795172e+04,Ejido Tehuantepec
3,p001220,f0066757,2.564389e+04,Ejido México
5,p001291,f0067447,1.819546e+04,Ejido Tlaxcala
7,p001476,f0067318,1.147236e+06,Tijuana
9,p001501,f0067126,2.558126e+04,Ingeniero Luis B. Sánchez
...,...,...,...,...
6640,p044376,f0099963,1.323658e+04,Las Ruedas
6641,p044388,f0092396,8.029428e+04,Viedma
6643,p044425,f0099582,3.948798e+03,Metri
6646,p044440,f0098009,2.974498e+03,Aulén
